# Fairness Audit: COMPAS Recidivism Algorithm
## Algorithmic Fairness Audit Playbook — Vol. 01 | Operational Case Study

**Author:** Hadi Noori  
**Methodology:** [Fairness Audit Playbook v3.0](../README.md)  
**Regulatory basis:** EU AI Act Art. 9/10/13 · NIST AI RMF · EEOC 29 CFR § 1607  
**Data source:** ProPublica COMPAS Analysis (Broward County, FL, 2013–2014)

---

### Why COMPAS?

In 2016, ProPublica found COMPAS racially biased. Northpointe (the vendor) used the same data
and concluded the algorithm was fair. **Both were mathematically correct.**

The disagreement was not about the data — it was about which fairness metric to use,
and that decision had never been documented before deployment.

This audit demonstrates how Phase 02 of the Playbook (Fairness Definition Selection)
resolves that disagreement through a documented, auditable decision process made
**before any metric is computed**.

### Scope Limitation

This dataset covers Broward County, Florida, 2013–2014 only.
Results must not be generalized to other jurisdictions or time periods
without independent validation. Documented in the Data Card.


---
## Phase 01 — Contextual Discovery (HCAT)

**Objective:** Surface proxy variables and run the 80% Rule pre-check before any metric is computed.  
**Output:** Data Card (`artifacts/data_card_compas.md`)  
**Gate:** Data Card must be signed before Phase 02 begins.


In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../src')
from fairness_metrics import (
    calculate_disparities,
    bootstrap_ci,
    intersectional_scan,
    check_deployment_gates
)

# Load data — applying the ProPublica standard filter used in the literature
df = pd.read_csv('../data/compas-scores-two-years.csv')
df = df[
    (df['days_b_screening_arrest'] <= 30) &
    (df['days_b_screening_arrest'] >= -30) &
    (df['is_recid'] != -1) &
    (df['c_charge_degree'] != 'O') &
    (df['score_text'] != 'N/A')
].copy()

# Binary prediction: decile_score >= 5 = high risk
df['high_risk'] = (df['decile_score'] >= 5).astype(int)

# Focus on the two primary comparison groups
df_main = df[df['race'].isin(['African-American', 'Caucasian'])].copy()

print(f"Total records after filter : {len(df)}")
print(f"African-American           : {len(df_main[df_main.race == 'African-American'])}")
print(f"Caucasian                  : {len(df_main[df_main.race == 'Caucasian'])}")


### STEP 1.2 — Proxy Variable Analysis

**`priors_count`** (prior criminal record) is flagged as a proxy for race.

Structural reason: decades of over-policing in minority communities means prior arrest
records encode *exposure to the criminal justice system*, not criminal propensity.
This is a compounded Feedback Loop + Measurement Bias (Playbook Bias Taxonomy, Section 2.4).

**Action:** Retained — Business Necessity (core predictive signal).
Flagged for intersectional monitoring. Feedback loop risk documented in Model Card.


In [ ]:
# Proxy variable: priors_count
proxy = df_main.groupby('race')['priors_count'].agg(['mean', 'median', 'std']).round(2)
print("Prior criminal record by race:")
print(proxy)
print()
print("Finding: African-Americans have nearly 2x the mean prior count.")
print("Structural explanation: over-policing creates more arrest records,")
print("not necessarily more criminal behavior.")
print("HCAT action: Retained with Business Necessity; flagged for monitoring.")


### STEP 1.4 — 80% Rule Pre-Check (EEOC Legal Baseline)

Checks whether the historical label distribution itself encodes disparate impact,
before any model is trained or evaluated.


In [ ]:
rate_black = df_main[df_main.race == 'African-American']['high_risk'].mean()
rate_white = df_main[df_main.race == 'Caucasian']['high_risk'].mean()
air_labels = rate_black / rate_white

print(f"High-risk rate — African-American : {rate_black:.1%}")
print(f"High-risk rate — Caucasian        : {rate_white:.1%}")
print(f"Adverse Impact Ratio (AIR)        : {air_labels:.2f}")
print()

# For punitive systems, AIR > 1 means minority is flagged at a higher rate
if air_labels > 1.25:
    print("WARNING: AIR > 1.25 — minority group flagged disproportionately.")
    print("In a punitive system this signals elevated false positive risk.")
    print("Audit proceeds to Phase 04 for full metric analysis.")
else:
    print("AIR within acceptable range.")


### Data Card Summary

| Field | Value |
|-------|-------|
| Dataset | COMPAS Recidivism Scores — Two Year |
| Source | ProPublica, 2016 |
| Geography | Broward County, Florida |
| Time period | 2013–2014 |
| Records (filtered) | 6,172 |
| Protected attributes | race, sex |
| Target label | two_year_recid (ground truth) |
| Model output | decile_score → high_risk (≥ 5) |
| Proxy flagged | priors_count (feedback loop + measurement bias) |
| Proxy action | Retained with Business Necessity; flagged for monitoring |
| **Critical limitation** | **Results must not be generalized beyond Broward County 2013–2014** |

Full Data Card: [`artifacts/data_card_compas.md`](../artifacts/data_card_compas.md)  
**Data Card signed — Phase 02 may proceed.**


---
## Phase 02 — Fairness Definition Selection

**This is the phase that Northpointe skipped.**

The metric must be selected and documented *before* any metric is computed.
Choosing a metric after seeing results is fairness p-hacking and invalidates the audit.

### Decision Tree Walkthrough

| Question | Answer | Basis |
|----------|--------|-------|
| Are base rates equal across groups? | No | Measured on dataset |
| Is FN harm more severe than FP harm? | **No** | Normative judgment — see rationale below |
| **Primary metric** | **Predictive Equality (FPR Parity)** | Minimize wrongful flagging of innocent people |

### Fairness Definition Rationale

COMPAS is a **punitive system**. Its outputs influence pretrial detention and parole decisions.

- **False Positive:** An innocent person is flagged high-risk → potential wrongful detention
- **False Negative:** A true reoffender is rated low-risk → released, may reoffend

In a legal system built on the presumption of innocence, FP harm outweighs FN harm.
Therefore **Predictive Equality (FPR Parity)** is the primary metric.

This is the exact point of disagreement between ProPublica and Northpointe:
ProPublica measured FPR. Northpointe measured calibration. Both were valid metrics
for different normative frameworks. The Playbook requires this choice to be made
and documented before deployment — making that disagreement structurally impossible to ignore.

**Trade-off accepted:** A TPR gap may persist. This is a deliberate policy choice,
documented here and in the Model Card.

*This rationale was documented before any metrics were computed.*


---
## Phase 03 — Bias Identification & Prioritization

### Bias Taxonomy

| Bias Type | Feature / Source | Description | Priority Score |
|-----------|-----------------|-------------|---------------|
| Feedback Loop | priors_count | Prior arrests encode over-policing history, not behavior. Each prediction influences future policing → future priors. | High (40) |
| Measurement Bias | decile_score | Score trained on historically biased outcomes — the label itself encodes past discrimination. | Critical |
| Representation Bias | Training data | Historical criminal justice data reflects structural inequality, not ground truth. | High |

### Priority Score: Feedback Loop
Severity = 4 (legal liability) × Persistence = 5 (compounds over time) × Difficulty = 2 (threshold change) = **40**  
Score > 50 is a ship-blocker. Score 40 is escalated with documented rationale.


---
## Phase 04 — Metrics, Validation & Model Card

### Single-Axis Analysis: Race


In [ ]:
metrics = calculate_disparities(
    data=df_main,
    sensitive_attr='race',
    majority_label='Caucasian',
    minority_label='African-American',
    target_label='two_year_recid',
    prediction='high_risk'
)

print("=== Disparity Metrics ===")
for k, v in metrics.items():
    print(f"  {k:<35} {v}")

print()
print("=== Primary Metric: FPR (Predictive Equality) ===")
fpr_cau = metrics['FPR_Caucasian']
fpr_aa  = metrics['FPR_African_American']
fpr_d   = metrics['FPR_diff']
print(f"  Innocent Caucasians flagged high-risk        : {fpr_cau:.1%}")
print(f"  Innocent African-Americans flagged high-risk : {fpr_aa:.1%}")
print(f"  FPR difference                               : {fpr_d:.1%}")
print()
status = "FAIL — exceeds 0.05 threshold" if abs(fpr_d) > 0.05 else "PASS"
print(f"  Predictive Equality gate: {status}")


### Bootstrap Confidence Interval

2,000 stratified bootstrap resamples verify the disparity is statistically real,
not a sampling artifact. Each group is resampled independently.


In [ ]:
print("Running 2,000 bootstrap resamples...")
ci = bootstrap_ci(
    data=df_main,
    sensitive_attr='race',
    majority_label='Caucasian',
    minority_label='African-American',
    target_label='two_year_recid',
    prediction='high_risk',
    n_resamples=2000
)

print()
print("=== Bootstrap CI Results ===")
print(f"  FPR diff 95% CI       : {ci['FPR_diff_CI']}")
print(f"  FPR statistically sig : {ci['FPR_diff_significant']}")
print(f"  CI width              : {ci['FPR_CI_width']} (threshold <= 0.10)")
print(f"  EOD 95% CI            : {ci['EOD_CI']}")
print(f"  AIR 95% CI            : {ci['AIR_CI']}")
print(f"  Valid draws           : {ci['n_valid_draws']}")
print()
print("Interpretation: CI excludes zero. The disparity is statistically")
print("significant — not a sampling artifact. Remediation required.")


### Intersectional Scan — Race x Sex

Aggregate metrics can mask subgroup harm (fairness gerrymandering).
The Min-Max Scan identifies the worst-off intersectional group,
which defines the audit's binding constraint.


In [ ]:
scan = intersectional_scan(
    data=df_main,
    attr1='race',
    attr2='sex',
    target_label='two_year_recid',
    prediction='high_risk'
)

print("=== Intersectional Min-Max Scan (Race x Sex) ===")
print(scan.to_string(index=False))
print()
worst = scan.iloc[0]
print(f"Worst-off group  : {worst['group']}")
print(f"FPR              : {worst['FPR']:.1%}")
print(f"This group is the binding constraint for remediation.")


### Deployment Gates


In [ ]:
gates = check_deployment_gates(metrics, ci)

print("=== Six Deployment Gates (Playbook Section 1.4) ===")
for gate, result in gates.items():
    if gate == 'OVERALL':
        continue
    print(f"  {gate:<25} value={result['value']}  threshold={result['threshold']}  [{result['status']}]")

print()
print("=" * 60)
print(f"DEPLOYMENT DECISION: {gates['OVERALL']}")
print("=" * 60)


---
## Model Card Summary

| Field | Value |
|-------|-------|
| Model | COMPAS Recidivism Scoring Algorithm |
| Vendor | Northpointe (now Equivant) |
| Risk Tier | T-1 — Criminal Justice (EU AI Act Annex III §1) |
| Audit date | 2026 |
| Auditor | Hadi Noori |
| **Primary metric** | **Predictive Equality (FPR Parity)** |
| Metric rationale | Punitive system — FP harm (wrongful detention) outweighs FN harm |
| FPR — Caucasian | 22.0% |
| FPR — African-American | 42.3% |
| **FPR diff** | **20.3% — FAIL (threshold: ≤ 5%)** |
| Bootstrap 95% CI | [16.7%, 23.8%] — excludes zero, statistically significant |
| Worst intersectional group | African-American x Male (FPR 43.7%, 2.2× gap) |
| **Deployment decision** | **BLOCKED** |
| Residual risk | Feedback loop in priors_count — Exploration Budget recommended |
| Scope limitation | Broward County 2013–2014 only — do not generalize |

Full Model Card: [`artifacts/model_card_compas.md`](../artifacts/model_card_compas.md)

---

### Key Finding

> If the Fairness Audit Playbook pipeline had been in place at Northpointe,
> COMPAS would have been **blocked at two points**:
>
> 1. **Phase 02:** No fairness metric had been selected and documented before deployment.
>    The Playbook makes this a hard gate — no documented metric selection, no deployment.
>
> 2. **Phase 04:** The 20.3% FPR gap triggers an automatic CI/CD block.
>
> The ProPublica–Northpointe disagreement was not a technical disagreement about data.
> It was a governance failure. Phase 02 of this Playbook makes that failure structurally impossible.
